# PySpark drills 6: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-06-joins.ipynb)

Practice for [PySpark 6: joins](pyspark-06-joins.ipynb). Each exercise uses the `customers`, `orders`, `order_items` and `products` tables.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC), `order_items` (one row per order line; each order's lines add up to its amount) and `products` (one row per SKU; `SKU-4014` is missing, so one order line has no product): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing: an order line with no product row
], "sku string, category string, list_price decimal(8,2)")

## Exercise 1: Customers who never ordered

**Difficulty:** Easy · about 5 min

Return `customer_id` and `name` of customers with no orders, ordered by `customer_id`.

**Expected output**

| customer_id | name |
| --- | --- |
| 7 | Gamma Sports |
| 8 | Hotel Homeware |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").orderBy("customer_id")
result.show()

## Exercise 2: Revenue by category without losing lines

**Difficulty:** Medium · about 10 min

Compute line revenue (`qty * unit_price`) per product `category`. Lines whose SKU has no product row count under
`unknown`. Return `category` and `revenue`, ordered by `category`. The total must equal the revenue of all lines.

**Expected output**

| category | revenue |
| --- | --- |
| Garden | 498.90 |
| Kitchen | 258.50 |
| Outdoor | 520.00 |
| Pantry | 1340.25 |
| unknown | 41.40 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    order_items.join(products.select("sku", "category"), "sku", "left")
    .groupBy(F.coalesce("category", F.lit("unknown")).alias("category"))
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
    .orderBy("category")
)
result.show()

## Exercise 3: Match keys that may be null

**Difficulty:** Medium · about 10 min

Join `targets` to `actuals` on both `country` and `region`, where a null region means "the whole country" on both
sides and must match. Return `country`, `region`, `target` and `actual`, ordered by `country` then `region` with nulls
first.

**Sample input**

**`targets`**

| country | region | target |
| --- | --- | --- |
| DE | Berlin | 100 |
| DE | null | 500 |
| FR | null | 300 |

**`actuals`**

| country | region | actual |
| --- | --- | --- |
| DE | Berlin | 90 |
| DE | null | 480 |
| FR | null | 310 |

**Expected output**

| country | region | target | actual |
| --- | --- | --- | --- |
| DE | null | 500 | 480 |
| DE | Berlin | 100 | 90 |
| FR | null | 300 | 310 |

In [ ]:
# Sample data for this exercise.
targets = spark.createDataFrame([("DE", "Berlin", 100), ("DE", None, 500), ("FR", None, 300)], "country string, region string, target int")
actuals = spark.createDataFrame([("DE", "Berlin", 90), ("DE", None, 480), ("FR", None, 310)], "country string, region string, actual int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
t, a = targets.alias("t"), actuals.alias("a")
result = (
    t.join(a, (F.col("t.country") == F.col("a.country")) & F.col("t.region").eqNullSafe(F.col("a.region")))
    .select("t.country", "t.region", "t.target", "a.actual")
    .orderBy("country", F.col("region").asc_nulls_first())
)
result.show()

## Exercise 4: Segment at the time of the order

**Difficulty:** Hard · about 20 min

`segments` holds customer segment versions with a half-open validity interval `[valid_from, valid_to)`; a null
`valid_to` means current. For each of customer 1's and customer 2's orders, return `order_id` and the `segment` valid
at `order_ts` (null if none). Order by `order_id`.

**Sample input**

**`segments`**

| customer_id | segment | valid_from | valid_to |
| --- | --- | --- | --- |
| 1 | standard | 2024-01-01 00:00:00 | 2026-01-05 10:00:00 |
| 1 | premium | 2026-01-05 10:00:00 | null |
| 2 | standard | 2026-01-07 00:00:00 | null |

**Expected output**

| order_id | segment |
| --- | --- |
| 1 | standard |
| 2 | premium |
| 3 | premium |
| 4 | null |
| 5 | standard |

In [ ]:
# Sample data for this exercise.
segments = spark.createDataFrame(
    [(1, "standard", datetime(2024, 1, 1), datetime(2026, 1, 5, 10, 0)), (1, "premium", datetime(2026, 1, 5, 10, 0), None),
     (2, "standard", datetime(2026, 1, 7, 0, 0), None)],
    "customer_id int, segment string, valid_from timestamp, valid_to timestamp",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
o, s = orders.where("customer_id IN (1, 2)").alias("o"), segments.alias("s")
condition = (
    (F.col("o.customer_id") == F.col("s.customer_id"))
    & (F.col("o.order_ts") >= F.col("s.valid_from"))
    & (F.col("o.order_ts") < F.coalesce(F.col("s.valid_to"), F.lit("9999-12-31").cast("timestamp")))
)
result = o.join(s, condition, "left").select("o.order_id", "s.segment").orderBy("order_id")
result.show()

## Exercise 5: Fix a duplicated dimension before joining

**Difficulty:** Medium · about 15 min

`regions` has several versions per customer. Keep each customer's latest version (by `updated_at`), then attach the
region to every order of customers 1 to 3. Return `order_id` and `region`, ordered by `order_id`; the number of rows
must equal the number of those orders.

**Sample input**

**`regions`**

| customer_id | region | updated_at |
| --- | --- | --- |
| 1 | North | 2025-01-01 00:00:00 |
| 1 | West | 2026-01-01 00:00:00 |
| 2 | North | 2025-06-01 00:00:00 |
| 3 | South | 2025-01-01 00:00:00 |
| 3 | Central | 2025-03-01 00:00:00 |

**Expected output**

| order_id | region |
| --- | --- |
| 1 | West |
| 2 | West |
| 3 | West |
| 4 | North |
| 5 | North |
| 6 | Central |
| 7 | Central |

In [ ]:
# Sample data for this exercise.
regions = spark.createDataFrame(
    [(1, "North", datetime(2025, 1, 1)), (1, "West", datetime(2026, 1, 1)), (2, "North", datetime(2025, 6, 1)),
     (3, "South", datetime(2025, 1, 1)), (3, "Central", datetime(2025, 3, 1))],
    "customer_id int, region string, updated_at timestamp",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
from pyspark.sql import Window

latest = Window.partitionBy("customer_id").orderBy(F.desc("updated_at"))
current = regions.withColumn("rn", F.row_number().over(latest)).where("rn = 1").select("customer_id", "region")
result = (
    orders.where("customer_id IN (1, 2, 3)").join(current, "customer_id", "left")
    .select("order_id", "region").orderBy("order_id")
)
result.show()

**Next:** [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb)